In [10]:
import pandas as pd
import glob
import os

# Ruta a la carpeta raw
raw_data_path = '../data/raw/'

# Buscar todos los archivos de datos (excluyendo el de nomenclatura si es necesario)
all_files = glob.glob(os.path.join(raw_data_path, "datos_abiertos_*.csv"))

# Leer y concatenar todos los archivos especificando el encoding
df_list = [pd.read_csv(f, encoding='latin1') for f in all_files]
df_mibici = pd.concat(df_list, ignore_index=True)

# Revisar la información inicial
df_mibici.info()
df_mibici.head()

<class 'pandas.DataFrame'>
RangeIndex: 39078984 entries, 0 to 39078983
Data columns (total 10 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Viaje_Id            int64  
 1   Usuario_Id          int64  
 2   Genero              str    
 3   AÃ±o_de_nacimiento  float64
 4   Inicio_del_viaje    str    
 5   Fin_del_viaje       str    
 6   Origen_Id           int64  
 7   Destino_Id          int64  
 8   Año_de_nacimiento   float64
 9   A}äe_nacimiento     float64
dtypes: float64(3), int64(4), str(3)
memory usage: 2.9 GB


,Viaje_Id,Usuario_Id,Genero,AÃ±o_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,Año_de_nacimiento,A}äe_nacimiento
0,4601,10324,M,NaN,2014-12-01 00:33:47,2014-12-01 00:36:54,47,47,NaN,NaN
1,4604,10324,M,NaN,2014-12-01 01:06:54,2014-12-01 01:08:45,5,5,NaN,NaN
2,4628,3,F,1990.0,2014-12-01 09:47:20,2014-12-01 09:47:47,79,79,NaN,NaN
3,4631,102,M,1982.0,2014-12-01 09:48:23,2014-12-01 09:48:37,79,79,NaN,NaN
4,4632,3,F,1990.0,2014-12-01 09:48:46,2014-12-01 09:48:57,79,79,NaN,NaN


## Empezamos con los ajustes ##


In [13]:
# Combinar las tres columnas en una sola. coalesce toma el primer valor no nulo que encuentra.
df_mibici['anio_nacimiento'] = df_mibici['Año_de_nacimiento'].combine_first(df_mibici['AÃ±o_de_nacimiento']).combine_first(df_mibici['A}äe_nacimiento'])
df_mibici['anio_nacimiento'] = df_mibici['anio_nacimiento'].astype('Int64')

# Eliminar las columnas originales defectuosas
df_mibici = df_mibici.drop(columns=['Año_de_nacimiento', 'AÃ±o_de_nacimiento', 'A}äe_nacimiento'])

In [14]:
# Revisar la información
df_mibici.info()
df_mibici.head()

<class 'pandas.DataFrame'>
RangeIndex: 39078984 entries, 0 to 39078983
Data columns (total 8 columns):
 #   Column            Dtype
---  ------            -----
 0   Viaje_Id          int64
 1   Usuario_Id        int64
 2   Genero            str  
 3   Inicio_del_viaje  str  
 4   Fin_del_viaje     str  
 5   Origen_Id         int64
 6   Destino_Id        int64
 7   anio_nacimiento   Int64
dtypes: Int64(1), int64(4), str(3)
memory usage: 2.4 GB


,Viaje_Id,Usuario_Id,Genero,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,anio_nacimiento
0,4601,10324,M,2014-12-01 00:33:47,2014-12-01 00:36:54,47,47,<NA>
1,4604,10324,M,2014-12-01 01:06:54,2014-12-01 01:08:45,5,5,<NA>
2,4628,3,F,2014-12-01 09:47:20,2014-12-01 09:47:47,79,79,1990
3,4631,102,M,2014-12-01 09:48:23,2014-12-01 09:48:37,79,79,1982
4,4632,3,F,2014-12-01 09:48:46,2014-12-01 09:48:57,79,79,1990


In [15]:
# Convertir a datetime
df_mibici['Inicio_del_viaje'] = pd.to_datetime(
    df_mibici['Inicio_del_viaje'], 
    format='mixed', errors='coerce')

df_mibici['Fin_del_viaje'] = pd.to_datetime(
    df_mibici['Fin_del_viaje'], 
    format='mixed', errors='coerce')

In [16]:
df_mibici.info()
df_mibici.head()

<class 'pandas.DataFrame'>
RangeIndex: 39078984 entries, 0 to 39078983
Data columns (total 8 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Viaje_Id          int64         
 1   Usuario_Id        int64         
 2   Genero            str           
 3   Inicio_del_viaje  datetime64[us]
 4   Fin_del_viaje     datetime64[us]
 5   Origen_Id         int64         
 6   Destino_Id        int64         
 7   anio_nacimiento   Int64         
dtypes: Int64(1), datetime64[us](2), int64(4), str(1)
memory usage: 2.4 GB


,Viaje_Id,Usuario_Id,Genero,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,anio_nacimiento
0,4601,10324,M,2014-12-01 00:33:47,2014-12-01 00:36:54,47,47,<NA>
1,4604,10324,M,2014-12-01 01:06:54,2014-12-01 01:08:45,5,5,<NA>
2,4628,3,F,2014-12-01 09:47:20,2014-12-01 09:47:47,79,79,1990
3,4631,102,M,2014-12-01 09:48:23,2014-12-01 09:48:37,79,79,1982
4,4632,3,F,2014-12-01 09:48:46,2014-12-01 09:48:57,79,79,1990


In [17]:
# Columnas en minusculas

df_mibici.columns = [col.lower() for col in df_mibici.columns]

In [18]:
df_mibici.isnull().sum()

viaje_id                 0
usuario_id               0
genero              144508
inicio_del_viaje         0
fin_del_viaje            0
origen_id                0
destino_id               0
anio_nacimiento      81651
dtype: int64

In [21]:
df_mibici = df_mibici[df_mibici['fin_del_viaje'] >= df_mibici['inicio_del_viaje']]